# ODS 层 — 原始数据入仓

将黄色 + 绿色出租车原始 Parquet 写入 ODS 层，保留原始字段，追加 `source_type` 和 `ingestion_ts` 元数据列，按 `source_type / pickup_year / pickup_month` 分区存储。

## 1. 初始化 SparkSession

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-ODS-Ingest")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")  # only overwrite touched partitions
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE      = os.path.expanduser("~/tlc-spark-pipeline")
RAW_Y     = f"{BASE}/data/raw/yellow"
RAW_G     = f"{BASE}/data/raw/green"
RAW_FHV   = f"{BASE}/data/raw/fhv_hv"
ODS_Y     = f"{BASE}/data/warehouse/ods/yellow"
ODS_G     = f"{BASE}/data/warehouse/ods/green"
ODS_FHV   = f"{BASE}/data/warehouse/ods/fhv_hv"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
SOURCE      = os.environ.get("SOURCE", "all")   # yellow | green | fhv_hv | all
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}  SOURCE={SOURCE}")
print(f"Spark {spark.version} ready")


## 2. 黄色出租车 → ODS

In [ ]:
if SOURCE not in ("yellow", "all"):
    print(f"SOURCE={SOURCE} — skipping yellow")
else:
    raw_yellow = spark.read.parquet(f"{RAW_Y}/*{TARGET_DATE[:7]}*.parquet")
    ods_yellow = (
        raw_yellow
        .withColumn("source_type",   F.lit("yellow"))
        .withColumn("ingestion_ts",  F.current_timestamp())
        .withColumnRenamed("tpep_pickup_datetime",  "pickup_datetime")
        .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("pickup_year",  F.year("pickup_datetime"))
        .withColumn("pickup_month", F.month("pickup_datetime"))
        .withColumn("pickup_day",   F.dayofmonth("pickup_datetime"))
        .filter(F.to_date("pickup_datetime") == TARGET_DATE)
    )
    print(f"Yellow raw rows (month): {raw_yellow.count():,}")
    print(f"Yellow ODS rows (day):   {ods_yellow.count():,}")
    ods_yellow.printSchema()


In [ ]:
if SOURCE not in ("yellow", "all"):
    pass
else:
    (
        ods_yellow
        .write
        .partitionBy("pickup_year", "pickup_month", "pickup_day")
        .mode("overwrite")
        .parquet(ODS_Y)
    )
    print(f"Yellow ODS written to: {ODS_Y}")


## 3. 绿色出租车 → ODS

In [ ]:
if SOURCE not in ("green", "all"):
    print(f"SOURCE={SOURCE} — skipping green")
else:
    raw_green = spark.read.parquet(f"{RAW_G}/*{TARGET_DATE[:7]}*.parquet")
    ods_green = (
        raw_green
        .withColumn("source_type",   F.lit("green"))
        .withColumn("ingestion_ts",  F.current_timestamp())
        .withColumnRenamed("lpep_pickup_datetime",  "pickup_datetime")
        .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("pickup_year",  F.year("pickup_datetime"))
        .withColumn("pickup_month", F.month("pickup_datetime"))
        .withColumn("pickup_day",   F.dayofmonth("pickup_datetime"))
        .filter(F.to_date("pickup_datetime") == TARGET_DATE)
    )
    print(f"Green raw rows (month): {raw_green.count():,}")
    print(f"Green ODS rows (day):   {ods_green.count():,}")


In [ ]:
if SOURCE not in ("green", "all"):
    pass
else:
    (
        ods_green
        .write
        .partitionBy("pickup_year", "pickup_month", "pickup_day")
        .mode("overwrite")
        .parquet(ODS_G)
    )
    print(f"Green ODS written to: {ODS_G}")


## 4. FHV HV 网约车 → ODS

Uber/Lyft 等网约车，列名已为 `pickup_datetime`，补充元数据列即可。

In [ ]:
if SOURCE not in ("fhv_hv", "all"):
    print(f"SOURCE={SOURCE} — skipping fhv_hv")
else:
    raw_fhv = spark.read.parquet(f"{RAW_FHV}/*{TARGET_DATE[:7]}*.parquet")
    ods_fhv = (
        raw_fhv
        .withColumn("source_type",  F.lit("fhv_hv"))
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("pickup_year",  F.year("pickup_datetime"))
        .withColumn("pickup_month", F.month("pickup_datetime"))
        .withColumn("pickup_day",   F.dayofmonth("pickup_datetime"))
        .filter(F.to_date("pickup_datetime") == TARGET_DATE)
    )
    print(f"FHV HV raw rows (month): {raw_fhv.count():,}")
    print(f"FHV HV ODS rows (day):   {ods_fhv.count():,}")


In [ ]:
if SOURCE not in ("fhv_hv", "all"):
    pass
else:
    (
        ods_fhv
        .write
        .partitionBy("pickup_year", "pickup_month", "pickup_day")
        .mode("overwrite")
        .parquet(ODS_FHV)
    )
    print(f"FHV HV ODS written to: {ODS_FHV}")


## 4. 验证 ODS

In [ ]:
for source, path in [("yellow", ODS_Y), ("green", ODS_G), ("fhv_hv", ODS_FHV)]:
    if SOURCE not in (source, "all"):
        print(f"{source}: skipped (SOURCE={SOURCE})")
        continue
    df = spark.read.parquet(path)
    cnt = df.count()
    print(f"{source}: {cnt:,} rows total in ODS")
    df.groupBy("pickup_year", "pickup_month", "pickup_day").count() \
      .orderBy("pickup_year", "pickup_month", "pickup_day").show(10)

spark.stop()
print("ODS ingest complete.")
